# Notebook 21: Gravity Partition Function (Paper II, \u00a710.11)

$$Z_{\text{grav}} = \sum_{N=3}^{\infty} e^{-b(N)} Z_{\text{frozen}}(N) = 1.3304$$

In [ ]:
import sys, math
sys.path.insert(0, '../src')
from planetary_polygons.extensions.gravity_partition import (
    b_exact, gravity_partition_function, organize_by_field,
    assembly_map_table, selberg_contribution
)

## 1. Term-by-term computation

In [ ]:
Z_total, terms = gravity_partition_function(N_max=20)
print(f"{'N':>4} {'field':>18} {'b(N)':>10} {'exp(-b)':>14} {'Z_frozen':>14} {'Z_N':>14} {'cum%':>8}")
print("-" * 84)
cum = 0
for N, D, field, I_N, exp_I, Z_fr, term in terms:
    cum += term
    if term > 1e-20 * Z_total:
        print(f"{N:4d} {field:>18} {I_N:10.4f} {exp_I:14.6e} {Z_fr:14.6e} {term:14.6e} {cum/Z_total*100:7.2f}%")
print(f"\nZ = {Z_total:.10f}")
print(f"N=3,...,8 capture {sum(t[6] for t in terms[:6])/Z_total*100:.2f}%.")

## 2. Decomposition by trace field

In [ ]:
fields = organize_by_field(terms)
print(f"{'Field':>22} {'N values':>20} {'Z_field':>14} {'%':>8}")
print("-" * 68)
for fld in sorted(fields.keys(), key=lambda f: -sum(e[2] for e in fields[f])):
    entries = fields[fld]
    Z_f = sum(e[2] for e in entries)
    print(f"{fld:>22} {', '.join(str(e[0]) for e in entries):>20} {Z_f:14.6e} {Z_f/Z_total*100:7.2f}%")
print(f"\nQ dominates at {sum(e[2] for e in fields.get('Q',[]))/Z_total*100:.1f}%.")

## 3. Assembly map: $c = 12b(N) \approx N^2$

In [ ]:
rows = assembly_map_table(N_max=20)
print(f"{'N':>4} {'b(N)':>10} {'c=12b':>10} {'N\u00b2':>6} {'c-N\u00b2':>10}")
print("-" * 44)
for N, b, c, Nsq, diff in rows:
    print(f"{N:4d} {b:10.4f} {c:10.4f} {Nsq:6d} {diff:10.4f}")
print("\nc - N\u00b2 = N + O(1): boundary self-energy.")

## 4. Selberg zeta corrections

In [ ]:
print(f"{'N':>4} {'l\u2080':>14} {'Z_Selberg':>14} {'corr%':>8}")
print("-" * 42)
for N in range(5, 16):
    l0, Z_sel = selberg_contribution(N)
    print(f"{N:4d} {l0:14.6f} {Z_sel:14.6f} {(1-Z_sel)*100:7.2f}%")
print("\nN\u22658: <1%. N=5: ~15%.")

## 5. Convergence

In [ ]:
print("Tail fraction Z(N>K)/Z:\n")
for K in [3, 4, 5, 6, 8, 10, 12]:
    tail = sum(t[6] for t in terms if t[0] > K)
    print(f"  Z(N>{K:2d})/Z = {tail/Z_total:.2e}")
print("\nSuper-exponential: exp(-N\u00b2/12) kills the tail.")

## Summary

1. $Z = 1.3304$. 2. Q dominates (91%). 3. $c = N^2+O(N)$. 4. Selberg negligible for $N\ge8$. 5. 99.99% from $N\le8$.